# RAPID hydrograph review

Review observed gauge events, refreshed model baselines and individual collapse responses. This notebook diagnoses timing/shape and domain changes; predictor benchmarking remains separate. Updated 5 October 2026.

## Purpose and reading order

The original notebook only inspected zero/positive/negative collapse timing groups and their selected resistance domains. It did not validate against gauges and used historical runs. This version reads the October hydraulic refresh, adds observed/model overlays on verified actual UTC, and preserves those three inspection groups. A sampled zero means a timestamp tie, not an identical hydrograph. Interpolated timing is shown as a separate diagnostic; no per-river calibration or peak alignment is applied.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate
    raise FileNotFoundError("Could not locate the river-hierarchy repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
OUTPUT_ROOT = REPO_ROOT / "network_variants" / "output"
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))

from rapid_tools import (
    build_geometry_context_frame,
    build_multi_example_timing_frame,
    make_timing_scatter,
    supported_timing_x_metrics,
    supported_timing_y_metrics,
)
from rapid_tools.refreshed_analysis import build_refreshed_analysis_frame, DEFAULT_REFRESH_ROOT
from rapid_tools.hydrograph_review import gauge_overlay, base_directory, event_clock
import plotly.graph_objects as go
import json
REFRESH_ROOT = DEFAULT_REFRESH_ROOT

## Configuration and input assembly

The cells below load the completed experiments and form the single analysis population used by every investigation section. Run them before using the review cells.


In [ ]:
EXAMPLE_IDS = ["02","03","05","11","13","15","21"]

# matched_parent_vs_network: fraction of the full network represented by the matched parent corridor
# child_vs_matched_parent: fraction of the matched parent corridor retained by the child object
LENGTH_MODE = "matched_parent_vs_network"

# For complete closed compounds, the exposure factor is 1 because Lambda_matched
# already represents the whole compound; only nested/unit domains use routed width share.
PRIMARY_X_METRIC = "len_fraction_width_weighted_log_lambda"
# Model-relative change: outlet-minus-input peak lag, normalized by the parent modeled lag.
PRIMARY_Y_METRIC = "lag_to_inflow_peak_pct_shift_vs_base"

# Optional sensitivity filter. Keep False for the complete result set.
EXCLUDE_NETWORK_EDGE_COMPOUNDS = True

# IMPORTANT — CHECK MATCHED-BOUNDARY VALIDITY FOR ALL NETWORKS BEFORE INTERPRETING LAMBDA-MATCHED RESULTS.
# EXCLUDE OR SEPARATELY FLAG ROWS WITH boundary_relation != 'exact_match': THEY ARE NOT STRICT WHOLE-OBJECT RESISTANCE COMPARISONS.

## Build the validated analysis population

Only states with the required RAPID and selected-domain equivalent-resistance records enter `plot_df`. The following cell is the shared input contract for all three response classes.


In [ ]:
analysis_df = build_refreshed_analysis_frame(EXAMPLE_IDS, refresh_root=REFRESH_ROOT, geometry_root=OUTPUT_ROOT, length_mode=LENGTH_MODE)

for column in [PRIMARY_X_METRIC, PRIMARY_Y_METRIC, "Lambda_matched"]:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")

analysis_df["primary_pair_complete"] = (
    np.isfinite(analysis_df[PRIMARY_X_METRIC])
    & np.isfinite(analysis_df[PRIMARY_Y_METRIC])
)

coverage = (
    analysis_df.groupby(["example_id", "mode"], dropna=False)
    .agg(
        states=("state_id", "nunique"),
        resistance_available=("Lambda_matched", lambda s: np.isfinite(s).sum()),
        timing_available=(PRIMARY_Y_METRIC, lambda s: np.isfinite(s).sum()),
        primary_pairs=("primary_pair_complete", "sum"),
    )
    .reset_index()
)
display(coverage)
display(analysis_df["plot_category"].value_counts(dropna=False).rename("states").to_frame())

## Apply the population filter

This applies the configured edge-compound filter and produces `plot_df`, the common state-level table used below. No association screening or model-selection figures are retained here.


In [ ]:
plot_df = analysis_df.loc[~analysis_df.technical_alias].copy()

if EXCLUDE_NETWORK_EDGE_COMPOUNDS:
    max_order = plot_df.groupby("example_id")["compound_downstream_order"].transform("max")
    is_edge_compound = (
        plot_df["compound_downstream_order"].notna()
        & ((plot_df["compound_downstream_order"] <= 1) | (plot_df["compound_downstream_order"] >= max_order))
    )
    plot_df = plot_df.loc[~is_edge_compound].copy()

plot_df = plot_df.loc[plot_df["primary_pair_complete"]].copy()
print(f"Plotting {len(plot_df)} states from {plot_df['example_id'].nunique()} examples.")

## Gauge hydrographs and optional model overlay

Select any or all cases. `INCLUDE_MODEL=False` gives gauges only; plot buttons also toggle model visibility. UTC is recovered from the original forcing after verifying identical time/flow arrays. No alignment to the observed downstream peak is performed. Set `NORMALIZE=False` to compare actual discharge; normalization is explicitly shape-only. Peak tables refer to the quality-sheet observation window; a model peak outside it requires inspecting the full model event.

If the model peak lies beyond the gauge review window, the display extends to include it; a dotted line marks the end of that window. Observations and peak tables retain the selected gauge window.

In [ ]:
GAUGE_EXAMPLES = EXAMPLE_IDS
INCLUDE_MODEL = True
NORMALIZE = False
gauge_peak_tables = []
for example in GAUGE_EXAMPLES:
    fig, table = gauge_overlay(example, include_model=INCLUDE_MODEL, normalize=NORMALIZE)
    fig.show()
    gauge_peak_tables.append(table)
display(pd.concat(gauge_peak_tables, ignore_index=True))

## Parent and child network states

Case-by-case hydrograph interpretation is recorded in **`RAPID/reviews/hydrograph_match_review.xlsx`** (review dated 5 October 2026). Use this section to inspect the actual parent and child geometry.

Choose one example and optionally a subset of `(mode, state_id)` keys from the table. `UNIT_SELECTION = None` selects all eligible states; **at most `MAX_NETWORK_PAIRS` (default 6) are drawn**. Any omitted states are counted explicitly. An empty list selects none. Technical aliases are excluded. The modes are `indep`, `compound`, and `island`.

Each pair shows the full base parent and the actual selected child on the same map extent. The matched comparison domains are highlighted. `ZOOM_TO_DOMAIN=False` shows the entire network. A compound's expanded hydraulic comparison domain can include more links than its original topological unit.


Each selected pair also has a **parent–child hydrograph** using the same example, unit selection and upper limit. These are whole-network outlet responses to the selected collapse, not local unit-outlet discharges. Blue is the parent; dashed red is the child. The shared input forcing is optional (`SHOW_INPUT_FORCING`). Set `SHOW_NETWORK_MAPS=False` to show only hydrographs, or `SHOW_HYDROGRAPHS=False` for only maps. Saved UTC timestamps, discharge magnitudes and the full routed event are preserved; use Plotly zoom to inspect overlapping peaks.


In [ ]:
from rapid_tools.hydrograph_review import select_network_pairs, parent_child_network_figure, parent_child_hydrograph_figure

NETWORK_EXAMPLE = "15"
NETWORK_MODES = None  # All modes; or ["island"] to filter the table before choosing a row.
MAX_NETWORK_PAIRS = 1
ZOOM_TO_DOMAIN = True
SHOW_NETWORK_MAPS = True
SHOW_HYDROGRAPHS = True
SHOW_INPUT_FORCING = True

available_units = analysis_df.loc[
    analysis_df.example_id.eq(NETWORK_EXAMPLE) & ~analysis_df.technical_alias,
    ["mode", "state_id", "target_id", "hydraulic_domain_type", "parent_domain_n_links", "child_domain_n_links"],
].drop_duplicates(["mode", "state_id"]).sort_values(["mode", "state_id"])
if NETWORK_MODES is not None:
    available_units = available_units[available_units["mode"].isin(NETWORK_MODES)]
print(f"{len(available_units)} available units/states; first 30 keys shown below. Full table: available_units.")
display(available_units.head(30))

# Select the third row of the freshly built table (zero-based position 2).
# Each key needs BOTH mode and state_id, even when NETWORK_MODES is set.
selected_unit = available_units.iloc[2]
# UNIT_SELECTION = [(selected_unit["mode"], selected_unit["state_id"])]
UNIT_SELECTION = [
    ("compound", "CB001_bubble_4"),
    ("island", "island_15_194")
]
# UNIT_SELECTION =None

# Alternatively: UNIT_SELECTION = None for all eligible states, or [] for none.
selected_networks, omitted_networks = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Plotting {len(selected_networks)} pairs. Upper limit: {MAX_NETWORK_PAIRS}. Omitted by limit: {len(omitted_networks)}.")
for _, selected_network in selected_networks.iterrows():
    if SHOW_NETWORK_MAPS:
        parent_child_network_figure(selected_network, zoom_to_domain=ZOOM_TO_DOMAIN).show()
    if SHOW_HYDROGRAPHS:
        parent_child_hydrograph_figure(selected_network, show_forcing=SHOW_INPUT_FORCING).show()


## Multiple collapse hydrographs in one graph

Uses the same `NETWORK_EXAMPLE`, `NETWORK_MODES`, `UNIT_SELECTION`, `MAX_NETWORK_PAIRS` and `SHOW_INPUT_FORCING` controls above. The parent is drawn once, with one line per selected collapse. `None` selects all eligible states up to the stated limit; click legend entries to isolate curves. These are whole-network outlet hydrographs on verified shared clocks, without peak alignment. The selection table preserves mode/state keys, including deliberately selected aliases; coincident lines are possible.


In [ ]:
# Reload helpers so this cell also works in an already-running kernel.
import importlib
import rapid_tools.hydrograph_review as hydrograph_review
importlib.reload(hydrograph_review)
from rapid_tools.hydrograph_review import collapse_hydrograph_overlay, select_network_pairs

overlay_networks, overlay_omitted = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Overlay: {len(overlay_networks)} collapse states; limit {MAX_NETWORK_PAIRS}; omitted {len(overlay_omitted)}.")
display(overlay_networks[["mode", "state_id", "target_id", "hydraulic_domain_type"]])
if len(overlay_networks):
    collapse_hydrograph_overlay(overlay_networks, show_forcing=SHOW_INPUT_FORCING).show()
else:
    print("No collapse states selected.")


## Shared review helpers

These helpers define the three lag classes, compact distributions, stable review indices, and the reusable state inspector.

In [ ]:
# import plotly.express as px

# # Shared state-level review frame. Timing classes are mutually exclusive and use
# # the stored percentage shift relative to the base state.
# investigation_df = plot_df.copy()
# lag_metric = "lag_to_inflow_peak_pct_shift_vs_base"
# EQ_METRIC = "Lambda_matched"
# investigation_df[lag_metric] = pd.to_numeric(investigation_df[lag_metric], errors="coerce")
# investigation_df[EQ_METRIC] = pd.to_numeric(investigation_df[EQ_METRIC], errors="coerce")
# investigation_df["neg_log10_lambda"] = -np.log10(investigation_df[EQ_METRIC].where(lambda s: s > 0))

# compact_columns = [
#     "review_index", "example_id", "mode", "state_id", "target_id", "plot_category",
#     lag_metric, EQ_METRIC, "neg_log10_lambda", "hydraulic_domain_type",
#     "parent_domain_n_links", "child_domain_n_links", "length_fraction",
#     "width_fraction_of_total",
# ]

# def _lag_subset_index_plot(frame, *, title, index_value):
#     fig = px.scatter(
#         frame, x="review_index", y=lag_metric, color="mode", symbol="plot_category",
#         hover_data=[c for c in compact_columns if c in frame.columns and c != "review_index"],
#         template="plotly_white", title=title,
#     )
#     fig.update_traces(marker=dict(size=10, line=dict(width=0.7, color="black")))
#     fig.add_hline(y=0, line_dash="dash", line_color="grey")
#     fig.add_vline(x=index_value, line_dash="dot", line_color="black")
#     fig.update_xaxes(dtick=1, title="subset index (change the REVIEW_INDEX below)")
#     fig.update_yaxes(title="lag to inflow peak shift vs base [%]")
#     return fig

# def _eq_distribution(frame, *, title):
#     fig = px.histogram(
#         frame, x=EQ_METRIC, color="mode", facet_col="plot_category",
#         barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title=title,
#         hover_data=[c for c in ["example_id", "state_id", "target_id", lag_metric] if c in frame],
#     )
#     fig.update_xaxes(type="log", title="Lambda_K (selected hydraulic domain; log scale)")
#     fig.update_yaxes(title="state count")
#     return fig



## Shared state inspector

The inspector is called by each section below. It shows the selected hydraulic domain when available; otherwise it explicitly falls back to the full-network geometry view.

In [ ]:
# import geopandas as gpd
# from plotly.subplots import make_subplots
# def _add_link_layer(fig, links, *, name, color, width, opacity=1.0, row=1, col=1):
#     xs, ys = [], []
#     for geometry in links.geometry.dropna():
#         geometries = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
#         for line in geometries:
#             x, y = line.xy
#             xs.extend(x); ys.extend(y); xs.append(None); ys.append(None)
#     if xs:
#         fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color=color, width=width), opacity=opacity, name=name, hoverinfo="skip"), row=row, col=col)


# def inspect_collapsed_state(example_id, mode, state_id):
#     row=analysis_df[(analysis_df.example_id==example_id)&(analysis_df['mode']==mode)&(analysis_df.state_id==state_id)].iloc[0]
#     child_run=Path(row.refreshed_run_dir);parent_dir=base_directory(example_id)
#     def links(directory):
#         manifest=json.loads((directory/'hydraulic_context/hydraulic_context_manifest.json').read_text())
#         return gpd.read_file(manifest['input_paths']['directed_links'])
#     parent=links(parent_dir);child=links(child_run.parent)
#     pids=[int(x) for x in row.matched_base_link_ids.split(',')];cids=[int(x) for x in row.child_link_ids.split(',')]
#     fig=make_subplots(rows=1,cols=2,subplot_titles=['Refreshed comparison domain','Parent and child outlet'])
#     _add_link_layer(fig,parent,name='Parent network',color='#cbd5e1',width=1)
#     _add_link_layer(fig,parent[parent.id_link.isin(pids)],name='Parent domain',color='#2563eb',width=3)
#     _add_link_layer(fig,child[child.id_link.isin(cids)],name='Child domain',color='#dc2626',width=3)
#     for directory,name,color in [(parent_dir/'run','Parent','#2563eb'),(child_run,'Child','#dc2626')]:
#         h=pd.read_csv(directory/'outlet_hydrograph.csv');t,_=event_clock(example_id,h)
#         fig.add_trace(go.Scatter(x=t,y=h.q_outlet_cms,name=name,line=dict(color=color)),row=1,col=2)
#     fig.update_layout(title=f'{example_id} / {state_id}: {row.hydraulic_domain_type}',height=550,template='plotly_white')
#     fig.update_yaxes(scaleanchor='x',scaleratio=1,row=1,col=1)
#     fig.update_xaxes(title_text='Time (UTC)',row=1,col=2);fig.update_yaxes(title_text='Discharge (m³/s)',row=1,col=2);fig.show()
#     display(row[['Lambda_matched','domain_expanded','parent_domain_n_links','child_domain_n_links','peak_time_shift_hours','interpolated_peak_shift_hours','peak_interpolation_status','event_duration_change_pct','peak_excess_change_pct']].to_frame('value'))


## 1. Zero lag shift — timing ties

These states have exactly zero stored shift in lag to the inflow peak. The purpose is to assess their selected-domain equivalent-resistance distribution and inspect individual ties, not to treat them as delayed responses.

In [ ]:
# ZERO_LAG_REVIEW_INDEX = 2
# zero_lag_cases = investigation_df.loc[investigation_df[lag_metric].eq(0)].sort_values(
#     ["example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# zero_lag_cases.insert(0, "review_index", zero_lag_cases.index)
# if zero_lag_cases.empty:
#     raise ValueError("No zero lag-shift states were found.")
# if not 0 <= ZERO_LAG_REVIEW_INDEX < len(zero_lag_cases):
#     raise IndexError(f"ZERO_LAG_REVIEW_INDEX must be between 0 and {len(zero_lag_cases) - 1}")
# zero_lag_selected = zero_lag_cases.iloc[ZERO_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"zero_lag_states": [len(zero_lag_cases)], "all_positive_or_zero": [int(investigation_df[lag_metric].ge(0).sum())]}))
# _eq_distribution(zero_lag_cases, title="Zero lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(zero_lag_cases, title=f"Zero lag shift candidates (n={len(zero_lag_cases)})", index_value=ZERO_LAG_REVIEW_INDEX).show()
# display(zero_lag_selected[[c for c in compact_columns if c in zero_lag_selected.index]].to_frame("selected zero-lag state"))
# inspect_collapsed_state(str(zero_lag_selected.example_id), str(zero_lag_selected['mode']), str(zero_lag_selected.state_id))


## 2. Positive lag shift — delayed outlet peak

These states have a positive relative lag shift. This section shows the lag distribution and selected-domain resistance-ratio distribution, then lets you inspect one state at a time.

In [ ]:
# POSITIVE_LAG_REVIEW_INDEX = 0
# positive_lag_cases = investigation_df.loc[investigation_df[lag_metric].gt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# positive_lag_cases.insert(0, "review_index", positive_lag_cases.index)
# if positive_lag_cases.empty:
#     raise ValueError("No positive lag-shift states were found.")
# if not 0 <= POSITIVE_LAG_REVIEW_INDEX < len(positive_lag_cases):
#     raise IndexError(f"POSITIVE_LAG_REVIEW_INDEX must be between 0 and {len(positive_lag_cases) - 1}")
# positive_lag_selected = positive_lag_cases.iloc[POSITIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"positive_lag_states": [len(positive_lag_cases)], "lag_min_pct": [positive_lag_cases[lag_metric].min()], "lag_max_pct": [positive_lag_cases[lag_metric].max()]}))
# px.histogram(positive_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Positive lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(positive_lag_cases, title="Positive lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(positive_lag_cases, title=f"Positive lag shift candidates (n={len(positive_lag_cases)})", index_value=POSITIVE_LAG_REVIEW_INDEX).show()
# display(positive_lag_selected[[c for c in compact_columns if c in positive_lag_selected.index]].to_frame("selected positive-lag state"))
# inspect_collapsed_state(str(positive_lag_selected.example_id), str(positive_lag_selected['mode']), str(positive_lag_selected.state_id))


In [ ]:
import importlib
import rapid_tools.hydrograph_review as review
importlib.reload(review)

## 3. Negative lag shift — earlier outlet peak

These states have a negative relative lag shift. They are kept separate because their physical interpretation differs from delayed peak timing; use the same distributions and inspector to review them.

In [ ]:
# NEGATIVE_LAG_REVIEW_INDEX = 0
# negative_lag_cases = investigation_df.loc[investigation_df[lag_metric].lt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# negative_lag_cases.insert(0, "review_index", negative_lag_cases.index)
# if negative_lag_cases.empty:
#     raise ValueError("No negative lag-shift states were found.")
# if not 0 <= NEGATIVE_LAG_REVIEW_INDEX < len(negative_lag_cases):
#     raise IndexError(f"NEGATIVE_LAG_REVIEW_INDEX must be between 0 and {len(negative_lag_cases) - 1}")
# negative_lag_selected = negative_lag_cases.iloc[NEGATIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"negative_lag_states": [len(negative_lag_cases)], "lag_min_pct": [negative_lag_cases[lag_metric].min()], "lag_max_pct": [negative_lag_cases[lag_metric].max()]}))
# px.histogram(negative_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Negative lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(negative_lag_cases, title="Negative lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(negative_lag_cases, title=f"Negative lag shift candidates (n={len(negative_lag_cases)})", index_value=NEGATIVE_LAG_REVIEW_INDEX).show()
# display(negative_lag_selected[[c for c in compact_columns if c in negative_lag_selected.index]].to_frame("selected negative-lag state"))
# inspect_collapsed_state(str(negative_lag_selected.example_id), str(negative_lag_selected['mode']), str(negative_lag_selected.state_id))


## Peak-window ambiguity across all 16 cases — 5 October 2026

This audit leaves routing and primary metrics unchanged. Exact maxima, near-peak blocks and normalization sensitivities are retained separately. A maximum **span may contain gaps**; its midpoint is only a diagnostic. The 99/99.5% discharge windows are not confidence intervals or event-duration thresholds. Case 05’s green window is the user-proposed earlier feature, not an adopted replacement peak or proof of lateral inflow. A representative peak rule must be independent of model agreement.

Changing observed peak choices affects validation and gauge-scaled speed; changing the forcing reference time also rescales model percentage lag. Absolute parent–child timing remains unchanged when only those references change.

In [ ]:
# # The Excel workbook is now the primary case-review record.
# from IPython.display import FileLink
# display(FileLink(str(REPO_ROOT / "RAPID/reviews/hydrograph_match_review.xlsx")))
